# FAMPNN Inverse Folding

![FAMPNN Inverse Folding](https://proto-bio.github.io/proto-assets/images/tool/fampnn/hero.png)

This notebook demonstrates four functions of the FAMPNN tool. In the first section, we use `run_fampnn_sample` to design new amino acid sequences with simultaneous sidechain co-generation. In the second section, we use `run_fampnn_pack` to predict sidechain conformations for a fixed backbone and sequence. In the third section, we use `run_fampnn_score` to evaluate specific point mutations using full-atom log-likelihood ratios. In the fourth section, we use `run_fampnn_score_all_mutations` to exhaustively score every possible single amino acid substitution at every position.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("fampnn")
display_overview("fampnn")
display_docs_section("fampnn", "Background")

# FAMPNN

Released in 2025, FAMPNN (Full-Atom MPNN) is an inverse-folding model that designs a sequence for a fixed backbone while jointly generating all of its sidechain atoms. Earlier fixed-backbone models reason about sidechains only implicitly; FAMPNN models each residue's amino-acid identity and its sidechain conformation together, which improves both sequence recovery and sidechain packing. It can design sequences, pack sidechains onto a fixed sequence, and score mutations with full-atom context, each with a per-atom sidechain confidence.

FAMPNN ([Widatalla et al., 2025](https://doi.org/10.1101/2025.02.13.637498)) solves the full-atom inverse-folding problem: given a fixed protein backbone, it predicts both an amino-acid sequence that folds into it and the sidechain conformation of every residue. Most fixed-backbone designers reason about sidechain interactions only implicitly, through backbone geometry and sequence labels, even though the three-dimensional arrangement of sidechain atoms drives protein conformation, stability, and function.

Internally, FAMPNN learns a per-residue joint distribution over the discrete amino-acid identity and the continuous sidechain conformation, trained with a combined categorical cross-entropy and diffusion objective. Sequences are generated by iterative unmasking, starting from a fully masked state and revealing residues over several steps, and the sidechain atoms are produced by a per-token Euclidean diffusion process in each residue's local backbone frame. A confidence module predicts the per-atom sidechain error (pSCE) in angstroms, which correlates with true packing error both per atom and, averaged over a residue, per residue. Learning sequence and sidechains jointly is synergistic, improving both native-sequence recovery and sidechain packing relative to modeling the sequence alone.

The reference implementation is maintained at [richardshuai/fampnn](https://github.com/richardshuai/fampnn) and was developed at Stanford University.

## Available tools

In [2]:
display_available_tools("fampnn")

- **`run_fampnn_pack()`** — Pack protein sidechains using FAMPNN with per-atom confidence (pSCE)
- **`run_fampnn_sample()`** — Design protein sequences with full-atom sidechain co-generation using FAMPNN
- **`run_fampnn_score()`** — Score protein mutations with full-atom context using FAMPNN
- **`run_fampnn_score_all_mutations()`** — Score every possible single mutation at every position using FAMPNN

### `run_fampnn_sample`

FAMPNN complexes new protein sequences that are predicted to fold into a target backbone structure by iteratively unmasking sequence positions while simultaneously denoising sidechain coordinates. Each designed residue comes with a predicted sidechain error (pSCE) score that quantifies confidence in the sidechain placement. You can optionally fix specific residue positions to preserve functional sites, and condition on known sidechain coordinates at selected positions to anchor the design around experimentally determined conformations.

In [3]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNSampleInput,
    FAMPNNSampleConfig,
    FAMPNNStructureInput,
    run_fampnn_sample,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_sample")

# Load GFP as the target backbone
gfp_structure = get_gfp_structure()

# Design all chains
sample_input = FAMPNNSampleInput(
    inputs=[FAMPNNStructureInput(structure=gfp_structure)]
)

**Input** — `FAMPNNSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[proto_tools.tools.inverse_folding.fampnn.fampnn_sample.FAMPNNStructureInput]</code> | required | List of structure inputs for sequence design. |

In [5]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_sample")

# Configure sampling with custom parameters | see docs above for all fields
sample_config = FAMPNNSampleConfig(
    num_sequences_per_structure=2,
    temperature=0.1,
    num_steps=10,  # Use 100 for production runs
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

# Fix specific residue positions to preserve functional sites
constrained_input = FAMPNNSampleInput(
    inputs=[FAMPNNStructureInput(
        structure=gfp_structure,
        chains_to_redesign=["A"],
        fixed_positions={"A": [10, 15, 20, 25]},
        fixed_sidechain_positions={"A": [10, 15]},
    )]
)

**Config** — `FAMPNNSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution) |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_sequences_per_structure</code> | <code>int</code> | <code>1</code> | Total number of sequences to generate per input structure. |
| <code>batch_size</code> | <code>int &#124; None</code> | <code>None</code> | Number of sequences to process simultaneously on GPU. Defaults to num_sequences_per_structure. |
| <code>temperature</code> | <code>float</code> | <code>0.1</code> | Sampling temperature; lower = greedier, higher = more diverse |
| <code>model_variant</code> | <code>str</code> | <code>'0.3'</code> | FAMPNN checkpoint: '0.3' (design), '0.0' (packing), '0.3_cath' (scoring) |
| <code>num_steps</code> | <code>int</code> | <code>100</code> | Number of iterative unmasking steps for sequence design |
| <code>seq_only</code> | <code>bool</code> | <code>False</code> | If True, skip sidechain generation during sampling |
| <code>repack_last</code> | <code>bool</code> | <code>True</code> | Repack sidechains after final sequence is determined |
| <code>psce_threshold</code> | <code>float</code> | <code>0.3</code> | Only keep sidechains below this predicted error threshold during design |
| <code>scn_diffusion_steps</code> | <code>int</code> | <code>50</code> | Number of sidechain diffusion denoising steps |
| <code>scn_step_scale</code> | <code>float</code> | <code>1.5</code> | Step scale (eta) for sidechain diffusion |

In [6]:
# Run the sampling function
result = run_fampnn_sample(constrained_input, sample_config)

Running run_fampnn_sample [00:00]

In [7]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_sample")

# Print designed sequences and per-residue pSCE scores
for i, design_set in enumerate(result.design_sets):
    for j, design in enumerate(design_set.complexes):
        seq = design.chains[0].sequence
        display_seq = f"{seq[:60]}..." if len(seq) > 60 else seq
        avg_psce = design.metrics["avg_psce"]
        print(f"Structure {i}, Design {j + 1}: {display_seq}")
        print(f"  Length: {len(seq)} residues, Mean pSCE: {avg_psce:.3f} A")
        print(f"  Chains: {list(zip([c.id for c in design.chains], design.designed, strict=True))}")

**Output** — `FAMPNNSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>design_sets</code> | <code>list[proto_tools.tools.inverse_folding.fampnn.fampnn_sample.FAMPNNDesignSet]</code> | required | One FAMPNNDesignSet per input structure, in input order. |

Structure 0, Design 1: SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLV...
  Length: 227 residues, Mean pSCE: 1.286 A
  Chains: [('A', True)]
Structure 0, Design 2: SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLV...
  Length: 227 residues, Mean pSCE: 1.293 A
  Chains: [('A', True)]


### Feed a design into a structure predictor

`Complex` accepts a `FAMPNNDesign` directly via `chains=design`, so a sampled design can be handed straight to a downstream structure predictor.

In [8]:
from proto_tools.tools.structure_prediction.shared_data_models import Complex

design = result.design_sets[0].complexes[0]
complex_input = design
print(f"Built Complex from a FAMPNNDesign with {len(design.chains)} chain(s)")

Built Complex from a FAMPNNDesign with 1 chain(s)


### `run_fampnn_pack`

Given a backbone structure and its sequence, FAMPNN predicts sidechain coordinates using per-token Euclidean diffusion. This is useful when you have a designed or known sequence but need atomic-level sidechain placement for downstream analysis such as molecular dynamics or ligand docking. The B-factor column in the output PDB contains per-atom pSCE values, so coloring by B-factor in a molecular viewer directly shows sidechain confidence.

In [9]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNPackInput,
    FAMPNNPackConfig,
    run_fampnn_pack,
)

In [10]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_pack")

# Pack sidechains for the GFP structure
pack_input = FAMPNNPackInput(
    inputs=[FAMPNNStructureInput(structure=gfp_structure)]
)

**Input** — `FAMPNNPackInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[proto_tools.tools.inverse_folding.fampnn.fampnn_sample.FAMPNNStructureInput]</code> | required | List of structure inputs for sidechain packing. |

In [11]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_pack")

# Configure packing | see docs above for all fields
pack_config = FAMPNNPackConfig(
    num_samples_per_structure=3,
    batch_size=3,
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `FAMPNNPackConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_variant</code> | <code>str</code> | <code>'0.0'</code> | FAMPNN checkpoint: '0.0' recommended for packing, '0.3' for design |
| <code>num_samples_per_structure</code> | <code>int</code> | <code>1</code> | Number of packing samples per input structure. |
| <code>batch_size</code> | <code>int</code> | <code>16</code> | Number of samples to process simultaneously on GPU |
| <code>scn_diffusion_steps</code> | <code>int</code> | <code>50</code> | Number of sidechain diffusion denoising steps |
| <code>scn_step_scale</code> | <code>float</code> | <code>1.5</code> | Step scale (eta) for sidechain diffusion |

In [12]:
# Run the packing function
pack_result = run_fampnn_pack(pack_input, pack_config)

Running run_fampnn_pack [00:00]

In [13]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_pack")

# Print mean pSCE for each packing sample
for i, (pdbs, psce_list) in enumerate(zip(pack_result.packed_structures, pack_result.psce)):
    print(f"Structure {i}: {len(pdbs)} packing samples")
    for j, psce in enumerate(psce_list):
        mean_psce = sum(psce) / len(psce)
        print(f"  Sample {j + 1}: mean pSCE = {mean_psce:.3f} A")

**Output** — `FAMPNNPackingResult`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>packed_structures</code> | <code>list[list[proto_tools.entities.structures.structure.Structure]]</code> | required | Packed structures (outer=inputs, inner=samples) |
| <code>psce</code> | <code>list[list[list[float]]]</code> | required | Per-residue predicted sidechain error in Å (shape [n_inputs, n_samples, n_residues]) |

Structure 0: 3 packing samples
  Sample 1: mean pSCE = 1.280 A
  Sample 2: mean pSCE = 1.279 A
  Sample 3: mean pSCE = 1.279 A


### `run_fampnn_score`

FAMPNN can score specific point mutations by computing log-likelihood ratios in full-atom context. Mutations are specified in the format `WT_RESIDUE + 1_INDEXED_POSITION + MUTANT_RESIDUE` (for example, `"S1V"` means serine to valine at position 1). Multiple simultaneous mutations are joined with colons: `"S1V:G5L"`. Positive scores indicate that the mutation is favored over wild-type. Use the special string `"wt"` to include the wild-type as a reference point, which always scores 0.0.

In [14]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNScoreInput,
    FAMPNNScoreConfig,
    MutationInput,
    run_fampnn_score,
)

In [15]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_score")

# Score specific mutations (1-indexed positions)
score_input = FAMPNNScoreInput(
    inputs=[MutationInput(
        structure=gfp_structure,
        mutations=["S1V", "K2A", "wt"],
    )]
)

**Input** — `FAMPNNScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[proto_tools.tools.inverse_folding.fampnn.fampnn_score.MutationInput]</code> | required | List of mutation inputs, each with a structure and mutations. |

In [16]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_score")

# Configure mutation scoring | see docs above for all fields
score_config = FAMPNNScoreConfig(
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `FAMPNNScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_variant</code> | <code>str</code> | <code>'0.3_cath'</code> | FAMPNN checkpoint: '0.3_cath' recommended for scoring |
| <code>batch_size</code> | <code>int</code> | <code>16</code> | Number of mutations to score simultaneously on GPU. |
| <code>seq_only</code> | <code>bool</code> | <code>False</code> | If True, score without sidechain context |
| <code>scn_diffusion_steps</code> | <code>int</code> | <code>50</code> | Number of sidechain diffusion denoising steps |
| <code>scn_step_scale</code> | <code>float</code> | <code>1.5</code> | Step scale (eta) for sidechain diffusion |

In [17]:
# Run the scoring function
score_result = run_fampnn_score(score_input, score_config)

Running run_fampnn_score [00:00]

In [18]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_score")

# Print scores for each mutation
for mut, score in zip(score_result.results[0].mutations, score_result.results[0].scores):
    label = " (wild-type)" if mut == "wt" else ""
    print(f"{mut}: {score:+.4f}{label}")

**Output** — `FAMPNNScoreOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.inverse_folding.fampnn.fampnn_score.MutationScoreResult]</code> | required | Scoring results, one per input structure |

S1V: -4.9167
K2A: -0.1259
wt: +0.0000 (wild-type)


### `run_fampnn_score_all_mutations`

For a comprehensive mutational landscape, FAMPNN can exhaustively score every possible single amino acid substitution at every position in the protein. The result is a position-by-residue dictionary of log-likelihood ratios that can be used to identify stabilizing mutations, map evolutionary constraints, or guide directed evolution experiments.

In [19]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNScoreAllMutationsInput,
    FAMPNNScoreAllMutationsConfig,
    run_fampnn_score_all_mutations,
)

In [20]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_score_all_mutations")

# Score all single mutations for the GFP structure
all_mut_input = FAMPNNScoreAllMutationsInput(inputs=[gfp_structure])

**Input** — `FAMPNNScoreAllMutationsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[proto_tools.entities.structures.structure.Structure]</code> | required | List of structures to score all possible single mutations. |

In [21]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_score_all_mutations")

# Configure exhaustive scoring | see docs above for all fields
all_mut_config = FAMPNNScoreAllMutationsConfig(
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `FAMPNNScoreAllMutationsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_variant</code> | <code>str</code> | <code>'0.3_cath'</code> | FAMPNN checkpoint: '0.3_cath' recommended for scoring |
| <code>batch_size</code> | <code>int</code> | <code>16</code> | Number of positions to score simultaneously on GPU. |

In [22]:
# Run the exhaustive scoring function
all_mut_result = run_fampnn_score_all_mutations(all_mut_input, all_mut_config)

Running run_fampnn_score_all_mutations [00:00]

In [23]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_score_all_mutations")

# Find the top beneficial mutations across all positions
scores = all_mut_result.results[0].scores
all_scores = []
for pos_label, pos_scores in scores.items():
    wt_residue = pos_label[-1]
    for mut_aa, score in pos_scores.items():
        if mut_aa != wt_residue:
            all_scores.append((pos_label, mut_aa, score))

all_scores.sort(key=lambda x: x[2], reverse=True)

print(f"Scored {len(scores)} positions x 20 amino acids")
print()
print("Top 5 beneficial mutations:")
for pos, aa, s in all_scores[:5]:
    print(f"  {pos} -> {aa}: {s:+.4f}")

**Output** — `FAMPNNScoreAllMutationsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.inverse_folding.fampnn.fampnn_score_all_mutations.AllMutationsScoreResult]</code> | required | All-mutations scoring results, one per input structure |

Scored 227 positions x 20 amino acids

Top 5 beneficial mutations:
  217L -> E: +4.3040
  24H -> K: +4.2155
  167I -> V: +4.1633
  162K -> T: +4.0760
  143H -> S: +3.7119


## Export Results

Designed sequences can be exported to FASTA or JSON format. Packed sidechain structures can be exported as PDB files. Mutation scoring results can be exported to CSV or JSON format for downstream analysis.

In [24]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="fampnn_sequences", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'fampnn_sequences.fasta'}")

pack_result.export(name="fampnn_packed", export_path=output_dir, file_format="pdb")
print(f"Packed structures exported to {output_dir / 'fampnn_packed/'}")

score_result.export(name="fampnn_scores", export_path=output_dir, file_format="csv")
print(f"Mutation scores exported to {output_dir / 'fampnn_scores.csv'}")

all_mut_result.export(name="fampnn_all_mutations", export_path=output_dir, file_format="csv")
print(f"All-mutation scores exported to {output_dir / 'fampnn_all_mutations/'}")

Designed sequences exported to example_output/fampnn_sequences.fasta
Packed structures exported to example_output/fampnn_packed
Mutation scores exported to example_output/fampnn_scores.csv
All-mutation scores exported to example_output/fampnn_all_mutations
